In [4]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv
load_dotenv()

client = MongoClient(os.environ["MONGODB_URI"])
for coll_name in ["chunks_fixed", "chunks_recursive"]:
    coll = client["smartstudy"][coll_name]
    print(coll_name, "->", coll.distinct("source"), "| count:", coll.count_documents({}))

chunks_fixed -> ['InformationRetrieval.pdf', 'en-chapter-1.pdf', 'en-chapter-2.pdf', 'en-chapter-3.pdf'] | count: 2975
chunks_recursive -> ['InformationRetrieval.pdf', 'en-chapter-1.pdf', 'en-chapter-2.pdf', 'en-chapter-3.pdf'] | count: 3071


In [5]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv
load_dotenv()
client = MongoClient(os.environ["MONGODB_URI"])
doc = client["smartstudy"]["chunks_fixed"].find_one({"source": "en-chapter-1.pdf"})
print(doc["page"], type(doc["page"]))
print(doc["source"], type(doc["source"]))

1 <class 'int'>
en-chapter-1.pdf <class 'str'>


In [6]:
from qa_set import QA_SET
gs = QA_SET[0]["gold_sources"][0]
print(gs["page"], type(gs["page"]))
print(gs["source"], type(gs["source"]))

21 <class 'int'>
en-chapter-1.pdf <class 'str'>


In [12]:
!pip install google.api_core

Defaulting to user installation because normal site-packages is not writeable
  Using cached google_api_core-2.32.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached googleapis_common_protos-1.75.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached proto_plus-1.28.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
Using cached google_api_core-2.32.0-py3-none-any.whl (174 kB)
Using cached googleapis_common_protos-1.75.0-py3-none-any.whl (300 kB)
Using cached proto_plus-1.28.1-py3-none-any.whl (50 kB)
Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl (439 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)

  Attempting uninstall: requests

    Found existing installation: requests 2.32.5

    Uninstalling requests-2.32.5:

      Successfully uninstalled requests-2.32.5

   ---------------------------------------- 0/5 [requests]
   --------------------

In [16]:
from dotenv import load_dotenv
load_dotenv()
from store import get_embeddings, get_vector_store, get_retriever
from config import CONFIG, resolve_mongo_cfg

mongo_cfg = resolve_mongo_cfg(CONFIG)  # make sure CONFIG["chunking"]["strategy"] == "fixed"
print("Querying:", mongo_cfg["collection_name"])

embeddings = get_embeddings(CONFIG["embedding"]["model"])
vector_store = get_vector_store(mongo_cfg, embeddings)
retriever = get_retriever(vector_store, k=4)

docs = retriever.invoke("What did the Treaty of Lisbon rename the Treaty establishing the European Community as?")
for d in docs:
    print(d.metadata.get("source"), d.metadata.get("page"), "-", d.page_content[:80])

Querying: chunks_fixed
en-chapter-1.pdf 21 - which drafted the Treaty establishing a Constitution for Europe (Constitutional 
en-chapter-1.pdf 21 - of the European Union’ (TFEU) and the term ‘Community’ is replaced by ‘Union’ th
en-chapter-1.pdf 22 - areas of its attributed powers or to join an international organisation. Member 
en-chapter-1.pdf 21 - the Constitutional Treaty, the Treaty of Lisbon contains no article formally ens


In [17]:
print("Num docs returned:", len(docs))

Num docs returned: 4
